In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ---------- helper functions ----------

def bitplane_decomposition(img_gray):
    """
    img_gray : 8-bit grayscale image (numpy array)
    returns 8 bit-planes in a list: planes[0] = LSB, planes[7] = MSB
    """
    planes = []
    for k in range(8):
        plane = (img_gray >> k) & 1      # extract bit k
        planes.append(plane.astype(np.uint8))
    return planes

def reconstruct_from_3_LSB(planes):
    """
    planes: list of 8 bit-planes from bitplane_decomposition
    returns image reconstructed from bits 0,1,2
    """
    recon = (planes[0] * 1 +
             planes[1] * 2 +
             planes[2] * 4).astype(np.uint8)
    return recon

def make_montage(img, planes, recon, diff, title):
    """
    Display 3×4 montage:
    row1: original, recon(3 LSB), diff, blank
    row2: bit 7, bit 6, bit 5, bit 4
    row3: bit 3, bit 2, bit 1, bit 0
    """
    plt.figure(figsize=(10, 8))

    # row 1
    plt.subplot(3, 4, 1), plt.imshow(img, cmap='gray'), plt.title('Original')
    plt.axis('off')
    plt.subplot(3, 4, 2), plt.imshow(recon, cmap='gray'), plt.title('3 LSB Recon')
    plt.axis('off')
    plt.subplot(3, 4, 3), plt.imshow(diff, cmap='gray'), plt.title('Difference')
    plt.axis('off')
    plt.subplot(3, 4, 4), plt.axis('off')

    # row 2: bits 7–4
    for i, b in enumerate(range(7, 3, -1), start=5):
        plt.subplot(3, 4, i)
        plt.imshow(planes[b] * 255, cmap='gray')
        plt.title(f'Bit {b}')
        plt.axis('off')

    # row 3: bits 3–0
    for i, b in enumerate(range(3, -1, -1), start=9):
        plt.subplot(3, 4, i)
        plt.imshow(planes[b] * 255, cmap='gray')
        plt.title(f'Bit {b}')
        plt.axis('off')

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def process_image(path, prefix):
    """
    path   : filename of low-light or bright-light image
    prefix : string used when saving outputs (e.g. 'low', 'bright')
    """
    # read and convert to grayscale
    img_color = cv2.imread(path)
    img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

    # bit-plane decomposition
    planes = bitplane_decomposition(img_gray)

    # reconstruction from 3 LSBs
    recon = reconstruct_from_3_LSB(planes)

    # difference image
    diff = cv2.absdiff(img_gray, recon)

    # save individual images if you need
    cv2.imwrite(f'{prefix}_original.png', img_gray)
    cv2.imwrite(f'{prefix}_recon3LSB.png', recon)
    cv2.imwrite(f'{prefix}_difference.png', diff)
    for k, p in enumerate(planes):
        cv2.imwrite(f'{prefix}_bitplane_{k}.png', p * 255)

    # show montage
    make_montage(img_gray, planes, recon, diff, f'{prefix.upper()} image')


# ---------- main ----------

if __name__ == "__main__":
    # Replace these names with your actual files
    low_image_path    = "car_low.png"     # low-light image
    bright_image_path = "car_bright.png"  # bright-light image

    process_image(low_image_path, "low")
    process_image(bright_image_path, "bright")

error: OpenCV(4.12.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'
